# Filtering for SACAIR runs to capture results

In [1]:
import logging
from pathlib import Path

#locals
# from code.run_types import ResSet, RunRes
import pandas as pd

from src.resulting import RESULTS_DIR, load_config_and_find_runs, print_json
from src.run_types import GenInfo


def load_find(conf_path: Path) -> GenInfo:
    runs =  load_config_and_find_runs(
            conf_path,
            exclude=exclude_keys,
            extra_mods=additional_modifications,
            logging_level=logging.DEBUG
        )
    assert runs is not None
    return runs


In [2]:
acc_cuttoff = 0
avail_models = ["MViTv2_S", "MViTv2_S_16x4", "MViTv2_B_32x3", "MViTv2_S_e",]

additional_modifications = {
    "results": {"best_val_acc": lambda x: x > acc_cuttoff},
    "admin": {"model": lambda x: x in avail_models},
    "scheduler": lambda x: x is None
}

exclude_keys = [
    ["results", "test_shuff"],
    ["results", "check_name"],
]

In [ ]:
from collections.abc import Callable
from typing import Any


def _unpack_filters(filters: dict) -> tuple[list[list[str]], list[Callable[[Any], bool]]]:

    filter_key_sets: list[list[str]] = []
    criterions: list[Callable[[Any], bool]] = []
    
    for key, value in filters.items():
        key_set = [key]
        if isinstance(value, Callable):
            criterions.append(value)
            filter_key_sets.append(key_set) 
            continue
            
        elif isinstance(value, dict):
            sub_key_sets, crits =  _unpack_filters(value)
            for sublist in sub_key_sets:
                filter_key_sets.append(key_set + sublist)
            
            criterions.extend(crits)
        else:
            raise TypeError(f'value should be dict or Callable, instead got: {type(value)}')
        

    return filter_key_sets, criterions

In [16]:
filter_key_sets, criterions = _unpack_filters(additional_modifications)
print(filter_key_sets)
print(len(filter_key_sets) == len(criterions))

[['results', 'best_val_acc'], ['admin', 'model'], ['scheduler']]
True


In [ ]:
def _pop_keys(d: dict, keys: list[Any], default = None) -> dict:
    if len(keys) == 1:
        return d.pop(keys[0], default)
    
    parent = d
    for key in keys[:-1]:
        parent = parent[key]

    return parent.pop(keys[-1], default)


def _drop_keys(d: dict, keys: list[Any]) -> dict:
    """Drop a nested value from a dict and return the dict. 

    Args:
        d (dict): A dictionary to modify in place
        keys (list[Any]): List of keys in order to index. 

    Returns:
        dict: The reference the original dictionary
    """
    
    parent = d
    for key in keys[:-1]:
        parent = parent[key]

    parent.pop(keys[-1])
    
    return d

In [31]:
d = {str(i): x for i in range(ord('a'), ord('e')) for x in range(5)}
e = {x : str(i) for i in range(ord('a'), ord('e')) for x in range(5)}
y = {'D': d, 'E': e}
print(y)

{'D': {'97': 4, '98': 4, '99': 4, '100': 4}, 'E': {0: '100', 1: '100', 2: '100', 3: '100', 4: '100'}}


In [32]:
print(_drop_keys(y, ['D']))

{'E': {0: '100', 1: '100', 2: '100', 3: '100', 4: '100'}}


In [33]:
print(y)

{'E': {0: '100', 1: '100', 2: '100', 3: '100', 4: '100'}}


In [17]:
results_dir = RESULTS_DIR / 'sacair_2026'
runs_16_p = results_dir / 'config_16f_15p.toml'
runs_32_p = results_dir / 'config_32f_15p.toml'
assert runs_16_p.exists(), f"{runs_16_p} not found"
assert runs_32_p.exists(), f"{runs_32_p} not found"

## Parameters in common:

The only parameter that differs is the number of frames. In all cases, models trained on asl300 upward were initialised from the previous split. 

In [19]:
runs_16 = load_find(runs_16_p)
runs_32 = load_find(runs_32_p)

DEBUG resulting: {
    "training": {
        "batch_size_equivalent": 8
    },
    "optimizer": {
        "eps": 1e-05,
        "backbone_init_lr": 0.0001,
        "backbone_weight_decay": 0.001,
        "classifier_init_lr": 0.001,
        "classifier_weight_decay": 0.001
    },
    "model_params": {
        "drop_p": 0.5
    },
    "data": {
        "strict_size": true,
        "target_length": 16,
        "frame_size": 224,
        "train_augs": {
            "normalise": true,
            "temporal_aug": [
                {
                    "target_length": 16,
                    "max_wobble": 0,
                    "type": "og",
                    "randomise": false
                }
            ],
            "spatial_aug": [
                {
                    "frame_size": 224,
                    "type": "Random_crop"
                },
                {
                    "type": "HORIZONTAL_FLIP",
                    "p": 0.5
                }
            ]
        }

## Runs 16 frames:

Lets check how many runs there are that match that spec:

In [20]:
for run in runs_16['results']:
    print_json(run['admin'])
    print()
print(len(runs_16['results']))

{
    "model": "S3D",
    "dataset": "WLASL",
    "split": "asl2000",
    "save_path": "/home/luke/Code/SLR/src/runs/asl2000/S3D/exp049/checkpoints",
    "seed": 42,
    "exp_no": "049",
    "recover": false,
    "config_path": "/home/luke/Code/SLR/src/results/satnac_2026/S3D/config_16f_15p.toml",
    "weight_path": "/home/luke/Code/SLR/src/runs/asl1000/S3D/exp048/checkpoints/best.pth"
}

{
    "model": "S3D",
    "dataset": "WLASL",
    "split": "asl1000",
    "save_path": "/home/luke/Code/SLR/src/runs/asl1000/S3D/exp048/checkpoints",
    "seed": 42,
    "exp_no": "048",
    "recover": false,
    "config_path": "/home/luke/Code/SLR/src/results/satnac_2026/S3D/config_16f_15p.toml",
    "weight_path": "/home/luke/Code/SLR/src/runs/asl300/S3D/exp047/checkpoints/best.pth"
}

{
    "model": "S3D",
    "dataset": "WLASL",
    "split": "asl300",
    "save_path": "/home/luke/Code/SLR/src/runs/asl300/S3D/exp047/checkpoints",
    "seed": 42,
    "exp_no": "047",
    "recover": false,
    "confi

## Runs 32 frames:

Lets check how many runs there are that match that spec:

<!-- Note there were 3 extra asl100s. We will take exp007 because it is used as a starting point for the asl300 run -->


In [21]:
# runs_32['results'] = runs_32['results'][:-3] 
for run in runs_32['results']:
    print_json(run['admin'])
    print()
print(len(runs_32['results']))

{
    "model": "S3D",
    "dataset": "WLASL",
    "split": "asl2000",
    "save_path": "/home/luke/Code/SLR/src/runs/asl2000/S3D/exp050/checkpoints",
    "seed": 42,
    "exp_no": "050",
    "recover": false,
    "config_path": "/home/luke/Code/SLR/src/results/satnac_2026/S3D/config_32f_15p.toml",
    "weight_path": "/home/luke/Code/SLR/src/runs/asl1000/S3D/exp049/checkpoints/best.pth"
}

{
    "model": "S3D",
    "dataset": "WLASL",
    "split": "asl1000",
    "save_path": "/home/luke/Code/SLR/src/runs/asl1000/S3D/exp049/checkpoints",
    "seed": 42,
    "exp_no": "049",
    "recover": false,
    "config_path": "/home/luke/Code/SLR/src/results/satnac_2026/S3D/config_32f_15p.toml",
    "weight_path": "/home/luke/Code/SLR/src/runs/asl300/S3D/exp048/checkpoints/best.pth"
}

{
    "model": "S3D",
    "dataset": "WLASL",
    "split": "asl300",
    "save_path": "/home/luke/Code/SLR/src/runs/asl300/S3D/exp048/checkpoints",
    "seed": 42,
    "exp_no": "048",
    "recover": false,
    "confi

## Now we can compare the runs

In [22]:
avail_acc_types = ["top_k_average_per_class_acc", "top_k_per_instance_acc"]
acc_type = avail_acc_types[1]
set_name = 'test'

#### Need to modify the names of S3D or they all get mixed together

In [23]:
df_format = []
for res in runs_16['results']: # + runs_32['results'] :
    model_name = res['admin']['model']
    if model_name == 'S3D':
        model_name += '_16'
    df_format.append(
        {'model': model_name, 'subset': res['admin']['split']} | {k : v for k, v in res['results'][set_name][acc_type].items()}
    )

for res in runs_32['results'] :
    model_name = res['admin']['model']
    if model_name == 'S3D':
        model_name += '_32'
    df_format.append(
        {'model': model_name, 'subset': res['admin']['split']} | {k : v for k, v in res['results'][set_name][acc_type].items()}
    )


df = pd.DataFrame(df_format)

In [24]:
df = df.rename(columns={"top1": "Top-1", "top5": "Top-5", "top10": "Top-10"})
# df

In [25]:
df['Top-1'] = df['Top-1'].apply(lambda x: f'{x*100:.2f}')
df['Top-5'] = df['Top-5'].apply(lambda x: f'{x*100:.2f}')
df['Top-10'] = df['Top-10'].apply(lambda x: f'{x*100:.2f}')

In [26]:
subsets = ['asl100', 'asl300', 'asl1000', 'asl2000']
for set_name in subsets:
    subdf = df[df['subset'] == set_name]
    print(f'{set_name}'.capitalize())
    display(subdf.sort_values('Top-1', ascending=False))

Asl100


,model,subset,Top-1,Top-5,Top-10
37,MViTv2_B_32x3,asl100,78.29,90.31,95.74
22,MViTv2_B_32x3,asl100,77.91,91.09,95.74
31,MViTv2_B_32x3,asl100,77.52,93.41,96.51
32,MViTv2_B_32x3,asl100,74.42,92.25,96.12
12,MViTv2_S_16x4,asl100,73.64,91.47,96.12
38,MViTv2_B_32x3,asl100,73.26,92.64,95.74
35,S3D_32,asl100,64.73,86.82,92.25
36,S3D_32,asl100,63.18,89.53,93.80
33,S3D_32,asl100,60.08,86.43,91.86
40,S3D_32,asl100,59.30,86.43,92.64


Asl300


,model,subset,Top-1,Top-5,Top-10
20,MViTv2_B_32x3,asl300,71.26,90.42,94.01
30,MViTv2_B_32x3,asl300,67.07,88.77,92.22
11,MViTv2_S_16x4,asl300,60.93,88.02,92.51
16,S3D_32,asl300,52.25,80.84,88.32
2,S3D_16,asl300,46.71,75.75,84.88
25,S3D_32,asl300,40.12,64.97,76.50
7,S3D_16,asl300,35.33,64.82,76.05


Asl1000


,model,subset,Top-1,Top-5,Top-10
19,MViTv2_B_32x3,asl1000,62.15,86.46,92.11
29,MViTv2_B_32x3,asl1000,56.77,83.48,89.66
10,MViTv2_S_16x4,asl1000,51.12,79.37,86.19
15,S3D_32,asl1000,44.56,74.47,83.26
1,S3D_16,asl1000,41.52,72.81,81.24
6,S3D_16,asl1000,30.38,59.97,70.58
24,S3D_32,asl1000,28.46,57.94,69.72


Asl2000


,model,subset,Top-1,Top-5,Top-10
18,MViTv2_B_32x3,asl2000,50.09,81.35,87.63
28,MViTv2_B_32x3,asl2000,44.56,76.35,83.71
9,MViTv2_S_16x4,asl2000,39.91,70.61,79.65
4,MViTv2_S_16x4,asl2000,37.65,69.36,80.38
14,S3D_32,asl2000,37.06,69.50,79.19
0,S3D_16,asl2000,33.83,66.69,76.00
23,S3D_32,asl2000,26.71,56.10,67.11
5,S3D_16,asl2000,23.38,51.23,64.19
27,MViTv2_B_32x3,asl2000,0.14,0.59,1.01
